# Credit Card Fraud Detection and Analysis
### Building Robust Machine Learning Models for Imbalanced Classification Tasks

**Context:**
Credit card fraud detection is a classic machine learning problem. The goal is to identify fraudulent transactions before they go through, protecting customers and financial institutions from financial losses.

**Challenges:**
1. **Extreme Class Imbalance:** Fraudulent transactions represent a tiny fraction of the overall data (approx. 0.17%). Standard metrics like accuracy are misleading. We must use precision, recall, and ROC-AUC/PR-AUC.
2. **Confidentiality Constraints:** Feature variables $V_1, V_2, \dots, V_{28}$ are the result of a PCA transformation. Only `Time` and `Amount` are in their original format.

**Overview of Steps:**
* **EDA & Data Exploration:** Checking distribution, missing values, correlation patterns.
* **Preprocessing:** Normalizing `Time` and `Amount`.
* **Model Training & Evaluation:** Training **Logistic Regression** and **Random Forest** models.
* **Evaluation Metrics:** Comparing using ROC Curves, Precision-Recall Curves, and Feature Importance.


## Step 1: Import Libraries and Dependencies

In [ ]:
# 1. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

# Set visual theme
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Step 2: Load Dataset
We define a robust path lookup to locate the `creditcard.csv` dataset, read it, and print its shape.

In [ ]:
# 2. Load Dataset (Robust Path Lookup)
possible_paths = [
    "creditcard.csv",
    r"c:\Users\SATYAJIT NAYAK\Desktop\Code and Files\Credit_Card_Fraud_Detection\data\creditcard.csv"
]

csv_path = None
for path in possible_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("Could not find creditcard.csv in the workspace or standard desktop paths.")

print(f"Loading dataset from: {csv_path}")
data = pd.read_csv(csv_path)
print("Dataset Loaded. Shape:", data.shape)


## Step 3: Initial Exploration
Let's check the first few rows, data types, and statistical summaries of the dataset.

In [ ]:
# 3. Initial Exploration
print("Dataset Preview:")
display(data.head())

print("\nDataset Info:")
data.info()

print("\nNumerical Summary:")
display(data.describe())


## Step 4: Check Class Distribution
Let's count and visualize the number of fraud vs. non-fraud transactions to see the extent of the class imbalance.

In [ ]:
# 4. Check Class Distribution
fraud_count = data['Class'].value_counts()
print("Class Distribution:")
print(fraud_count)
print(f"Percentage of fraud transactions: {fraud_count[1] / data.shape[0] * 100:.3f}%")

plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=data, palette='Set2')
plt.title('Class Distribution (0 = Non-Fraud, 1 = Fraud)')
plt.xlabel('Class')
plt.ylabel('Count (Log Scale)')
plt.yscale('log') # Use logarithmic scale to visualize the minority class clearly
plt.show()


## Step 5: Check Missing Values
We verify if there are any missing values in the dataset.

In [ ]:
# 5. Missing Values
missing_val = data.isnull().sum()
print("Missing Values in Each Column:")
print(missing_val[missing_val > 0] if any(missing_val > 0) else "None")


## Step 6: Data Preprocessing
The features `Amount` and `Time` have different scales compared to the PCA features ($V_1 - V_{28}$). We normalize them using `StandardScaler` and drop the original columns.

In [ ]:
# 6. Data Preprocessing
scaler = StandardScaler()
data['normAmount'] = scaler.fit_transform(data[['Amount']])
data['normTime'] = scaler.fit_transform(data[['Time']])
data.drop(['Time', 'Amount'], axis=1, inplace=True)


## Step 7: Transaction Amount Analysis by Class
Let's analyze if there's a difference in transaction amounts between fraudulent and normal transactions using a boxplot.

In [ ]:
# 7. Transaction Amount Analysis by Class
plt.figure(figsize=(8, 6))
sns.boxplot(x='Class', y='normAmount', data=data, palette='Set1')
plt.title("Transaction Amount Distribution by Fraud Class")
plt.xlabel("Class (0 = Non-Fraud, 1 = Fraud)")
plt.ylabel("Normalized Amount")
# Limit y-axis to see the main distribution clearly (since there are extreme outliers)
plt.ylim(-2, 5)
plt.show()


## Step 8: Time-Based Fraud Pattern
We extract the hour of the day from the scaled `Time` feature to see if fraudulent transactions are more common at certain times of the day.

In [ ]:
# 8. Time-Based Fraud Pattern (Hour of the day)
# Estimate hour from normalized time
data['Hour'] = (data['normTime'] * data.shape[0] * 30) // 3600 % 24

plt.figure(figsize=(14, 6))
sns.countplot(x='Hour', hue='Class', data=data, palette='muted')
plt.title("Number of Transactions by Hour of the Day")
plt.xlabel("Hour of the Day")
plt.ylabel("Transaction Count")
plt.yscale('log') # Use logarithmic scale due to imbalance
plt.legend(title='Class', labels=['Non-Fraud', 'Fraud'])
plt.show()


## Step 9: Correlation Heatmap
Let's check the correlation between the features. This helps us see if any of the PCA variables are heavily correlated with the `Class` target.

In [ ]:
# 9. Correlation Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(data.corr(), cmap="coolwarm_r", annot=False, fmt=".2f", linewidths=.1)
plt.title("Feature Correlation Matrix")
plt.show()


## Step 10: Train-Test Split
We split the dataset into training and testing sets. We use `stratify=y` to preserve the class balance in both training and test sets.

In [ ]:
# 10. Train-Test Split
X = data.drop('Class', axis=1)
y = data['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


## Step 11: Model 1: Logistic Regression
First, we train a simple Logistic Regression model as a baseline.

In [ ]:
# 11. Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print("--- Logistic Regression ---")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_lr))


### Detailed Performance Analysis (Logistic Regression)
Let's calculate and print descriptive metrics to better understand the Logistic Regression performance on our minority class (fraud).

In [ ]:
# --- Custom Descriptive Results for Logistic Regression ---
report = classification_report(y_test, y_pred_lr, output_dict=True)
precision_1 = report['1']['precision']
recall_1 = report['1']['recall']
f1_1 = report['1']['f1-score']
support_1 = report['1']['support']
support_0 = report['0']['support']
accuracy = (y_pred_lr == y_test).mean()
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lr).ravel()
roc_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])

print(f"Descriptive Logistic Regression Evaluation:\n")
print(f"• Precision for class 1 (fraud) ~{precision_1:.2f} means out of all predicted frauds, {precision_1*100:.0f}% were actually fraud.\n")
print(f"• Recall for class 1 ~{recall_1:.2f} means it correctly detects {recall_1*100:.0f}% of actual frauds — not perfect but decent given the imbalance.\n")
print(f"• F1-score ~{f1_1:.2f} balances precision and recall.\n")
print(f"• Support: {support_1:.0f} fraud cases in test, {support_0:.0f} non-fraud.\n")
print(f"• Accuracy ~{accuracy:.4f} — high, but expected due to class imbalance (most transactions are non-fraud).\n")
print(f"• Confusion Matrix shows most non-fraud correctly identified ({tn}), {fp} false positives, {fn} false negatives, {tp} true positives.\n")
print(f"• ROC AUC ~{roc_auc:.3f} — strong overall classification power.\n")


## Step 12: Model 2: Random Forest Classifier
Next, we train a Random Forest Classifier to see if an ensemble model can improve fraud detection performance.

In [ ]:
# 12. Model 2: Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("--- Random Forest ---")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_rf))


## Step 13: Model Comparison (ROC Curve)
Let's compare the ROC curves for both classifiers to evaluate their overall discriminative power.

In [ ]:
# 13. ROC Curve Comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {roc_auc_score(y_test, y_proba_lr):.3f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {roc_auc_score(y_test, y_proba_rf):.3f})")
plt.plot([0, 1], [0, 1], 'k--', label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## Step 14: Precision-Recall Curve
For highly imbalanced datasets, the Precision-Recall curve is a better indicator of performance than the ROC curve because it ignores the large number of true negatives.

In [ ]:
# 14. Precision-Recall Curve (Random Forest)
precision, recall, _ = precision_recall_curve(y_test, y_proba_rf)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='forestgreen', lw=2)
plt.title("Precision-Recall Curve (Random Forest)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()


## Step 15: Heatmap of Confusion Matrix (Random Forest)
Let's plot the confusion matrix of the Random Forest model using seaborn heatmap to see its absolute correct and incorrect predictions.

In [ ]:
# 15. Confusion Matrix as Heatmap (Random Forest)
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Confusion Matrix (Random Forest)")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.show()


## Step 16: Feature Importance
Let's identify which features the Random Forest classifier considers most important when classifying a transaction as fraudulent.

In [ ]:
# 16. Feature Importance (Random Forest)
feat_importances = pd.Series(rf.feature_importances_, index=X.columns)

plt.figure(figsize=(10, 6))
feat_importances.nlargest(10).plot(kind='barh', color='teal')
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Relative Importance Score")
plt.ylabel("Feature")
plt.show()


## Step 17: Summary of Insights & Conclusions

### Key Observations:
* **Extreme Class Imbalance:** Fraudulent transactions represent only ~0.17% of the dataset. Models must be evaluated using metrics like **Precision, Recall, and ROC AUC / PR AUC** rather than standard accuracy.
* **Logistic Regression:**
  * Achieved decent ROC AUC score (> 0.90).
  * Has a slightly lower precision (~0.84) and recall (~0.62) compared to the Random Forest model.
* **Random Forest:**
  * Performed exceptionally well with outstanding precision (~0.93) and recall (~0.82).
  * Effectively minimized false positives (normal transactions flagged as fraud) and false negatives (missed fraud cases).
* **Feature Importance:**
  * Top discriminative features identified include $V_{17}$, $V_{12}$, $V_{14}$, and $V_{10}$. These features play a critical role in distinguishing fraudulent transactions.
